# LangGraph Fundamentals

This notebook introduces the core ideas behind LangGraph:

- a typed `StateGraph`
- nodes and conditional edges
- compiling and invoking a graph
- graph visualization
- LangSmith node spans with `@traceable`

LangGraph models workflows as graphs made of state, nodes, and edges. The docs say the main graph class is `StateGraph`, the state schema is commonly defined with `TypedDict` or a Pydantic model, and graphs must be compiled before use.


## Learning goals

By the end of this notebook, you should be able to:

1. Define graph state with a `TypedDict`.
2. Add nodes to a `StateGraph`.
3. Route with conditional edges.
4. Compile and invoke the graph.
5. Visualize the graph.
6. Trace node spans in LangSmith.


## 1) Install packages

```bash
pip install -U langgraph langsmith
```

LangSmith’s annotate-code guide says `LANGSMITH_TRACING` controls `@traceable`, and `LANGSMITH_PROJECT` sets the project name. 


In [1]:
%pip install -qU langgraph langsmith


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-tests 1.1.4 requires pytest<9.0.0,>=7.0.0, but you have pytest 9.0.3 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2) Configure LangSmith

Set these in your environment or `.env` file:

```env
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=your_key
LANGSMITH_PROJECT=langgraph-fundamentals
```


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))
print("LANGSMITH_API_KEY set:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LANGSMITH_PROJECT:", os.getenv("LANGSMITH_PROJECT"))


LANGSMITH_TRACING: true
LANGSMITH_API_KEY set: True
LANGSMITH_PROJECT: lcel-groq-demo


## 3) Import LangGraph pieces


In [3]:
from typing_extensions import TypedDict
from typing import Literal

from langgraph.graph import StateGraph, START, END
from langsmith import traceable


## 4) Define graph state

The LangGraph docs say graph state is commonly defined with `TypedDict`. For this simple lab we only need a few keys and no custom reducer. 


In [4]:
class GraphState(TypedDict):
    topic: str
    route: str
    draft: str
    final: str


## 5) Define the nodes

We will build a small workflow:

- `classify_topic`: decide which branch to follow
- `research_node`: create a concise technical draft
- `review_node`: create a more polished summary
- `finalize_node`: assemble the final answer

The docs describe nodes as functions that receive state, do work, and return updated state. 


In [5]:
@traceable(name="classify_topic", run_type="chain")
def classify_topic(state: GraphState) -> dict:
    topic = state["topic"].lower()
    if any(word in topic for word in ["rag", "retrieval", "vector", "chunk"]):
        route = "research"
    else:
        route = "review"
    return {"route": route}

@traceable(name="research_node", run_type="chain")
def research_node(state: GraphState) -> dict:
    topic = state["topic"]
    return {"draft": f"{topic}: focus on structure, retrieval, and practical implementation."}

@traceable(name="review_node", run_type="chain")
def review_node(state: GraphState) -> dict:
    topic = state["topic"]
    return {"draft": f"{topic}: emphasize clarity, workflow order, and final presentation."}

@traceable(name="finalize_node", run_type="chain")
def finalize_node(state: GraphState) -> dict:
    return {"final": f"Topic: {state['topic']} | Route: {state['route']} | Draft: {state['draft']}"}


## 6) Conditional edges

The LangGraph docs say conditional edges call a routing function after a node runs and then choose the next node based on the returned value. They also note that you should use either static edges or dynamic routing from the same node, not both.


In [6]:
def route_after_classify(state: GraphState) -> Literal["research", "review"]:
    return state["route"]

builder = StateGraph(GraphState)

builder.add_node("classify_topic", classify_topic)
builder.add_node("research_node", research_node)
builder.add_node("review_node", review_node)
builder.add_node("finalize_node", finalize_node)

builder.add_edge(START, "classify_topic")
builder.add_conditional_edges(
    "classify_topic",
    route_after_classify,
    {"research": "research_node", "review": "review_node"},
)
builder.add_edge("research_node", "finalize_node")
builder.add_edge("review_node", "finalize_node")
builder.add_edge("finalize_node", END)

graph = builder.compile()

print("Graph compiled successfully.")


Graph compiled successfully.


## 7) Invoke the graph

Graphs must be compiled before use. The docs explicitly say you must call `.compile()` before invoking a graph.


In [7]:
result = graph.invoke({
    "topic": "End-to-end RAG pipeline",
    "route": "",
    "draft": "",
    "final": "",
})

result


{'topic': 'End-to-end RAG pipeline',
 'route': 'research',
 'draft': 'End-to-end RAG pipeline: focus on structure, retrieval, and practical implementation.',
 'final': 'Topic: End-to-end RAG pipeline | Route: research | Draft: End-to-end RAG pipeline: focus on structure, retrieval, and practical implementation.'}

In [8]:
result_2 = graph.invoke({
    "topic": "General notebook cleanup",
    "route": "",
    "draft": "",
    "final": "",
})

result_2


{'topic': 'General notebook cleanup',
 'route': 'review',
 'draft': 'General notebook cleanup: emphasize clarity, workflow order, and final presentation.',
 'final': 'Topic: General notebook cleanup | Route: review | Draft: General notebook cleanup: emphasize clarity, workflow order, and final presentation.'}

## 8) Visualize the graph

The use-graph-api docs show that you can visualize a graph and convert it to Mermaid syntax with `app.get_graph().draw_mermaid()`. 


In [9]:
mermaid = graph.get_graph().draw_mermaid()
print(mermaid)


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify_topic(classify_topic)
	research_node(research_node)
	review_node(review_node)
	finalize_node(finalize_node)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify_topic;
	classify_topic -. &nbsp;research&nbsp; .-> research_node;
	classify_topic -. &nbsp;review&nbsp; .-> review_node;
	research_node --> finalize_node;
	review_node --> finalize_node;
	finalize_node --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 9) LangSmith node spans

LangSmith’s annotate-code docs say `@traceable` turns a function into a traced run, and nested traced functions become child spans automatically. That means each node in this notebook appears as its own span in LangSmith when tracing is enabled. 


In [10]:
demo = graph.invoke({
    "topic": "Retrieval augmented generation",
    "route": "",
    "draft": "",
    "final": "",
})

demo


{'topic': 'Retrieval augmented generation',
 'route': 'research',
 'draft': 'Retrieval augmented generation: focus on structure, retrieval, and practical implementation.',
 'final': 'Topic: Retrieval augmented generation | Route: research | Draft: Retrieval augmented generation: focus on structure, retrieval, and practical implementation.'}

## 10) A small mental model

Use this mapping:

- `State` = the shared notebook of the graph
- `Node` = one step that reads state and writes updates
- `Conditional edge` = a routing decision
- `Compile` = validate and prepare the graph
- `Invoke` = run the graph once
- `Traceable node` = a span in LangSmith


## 11) Common mistakes

1. Forgetting to compile the graph.
2. Mixing static edges and dynamic routing from the same node.
3. Returning the wrong keys from a node.
4. Using unclear state.
5. Forgetting to turn on LangSmith tracing when you want spans.

The LangGraph docs warn against mixing static edges with dynamic routing from the same node because both paths can run and make the graph harder to reason about.


## Key takeaways

- `StateGraph` is the main graph builder.
- `TypedDict` is the beginner-friendly way to define state.
- Conditional edges route the graph based on a function output.
- Graphs must be compiled before invocation.
- `draw_mermaid()` helps you visualize the graph.
- `@traceable` gives you LangSmith node spans and nested tracing.


## References

- Graph API overview: https://docs.langchain.com/oss/python/langgraph/graph-api
- Conditional edges: https://docs.langchain.com/oss/python/langgraph/graph-api#conditional-edges
- Visualize your graph: https://docs.langchain.com/oss/python/langgraph/use-graph-api#visualize-your-graph
- Annotate code: https://docs.langchain.com/langsmith/annotate-code
